In [1]:
import os
import json
import random
import re
import pandas as pd

os.makedirs("data", exist_ok=True)

mountains = [
    "Mount Everest", "K2", "Kangchenjunga", "Lhotse", "Makalu", 
    "Cho Oyu", "Dhaulagiri", "Manaslu", "Nanga Parbat", "Annapurna",
    "Gasherbrum I", "Broad Peak", "Gasherbrum II", "Shishapangma", "Denali",
    "Aconcagua", "Mount Kilimanjaro", "Elbrus", "Mount Blanc", "Matterhorn",
    "Mount Fuji", "Hoverla", "Gerlachovsky Stit", "Mount Cook", "Ben Nevis",
    "Mount Washington", "Mount Whitney", "Mount Rainier", "Pikes Peak", "Mount Olympus"
]

df_mountains = pd.DataFrame(mountains, columns=["mountain_name"])
df_mountains.to_csv("data/raw_mountains.csv", index=False)


positive_templates = [
    "We are planning an expedition to {mountain} next month.",
    "The summit of {mountain} is covered with thick ice.",
    "Many brave climbers tried to conquer {mountain} this year.",
    "Yesterday we read a fascinating article about {mountain}.",
    "The view from the top of {mountain} was absolutely breathtaking.",
    "Is {mountain} considered dangerous for beginner hikers?",
    "They spent three weeks trekking near {mountain}.",
    "Heavy snowfall forced the team to retreat from {mountain}.",
    "He achieved his lifelong dream of climbing {mountain}.",
    "The local weather around {mountain} can change in minutes."
]


negative_sentences = [
    "I love going to the ocean during summer holidays.",
    "Next week we have an important meeting with our client.",
    "She decided to buy a new laptop for her work.",
    "The coffee shop on the corner serves the best espresso.",
    "Learning Data Science requires consistency and patience.",
    "We walked through the ancient streets of the city.",
    "The weather today is sunny and very pleasant."
]


def tokenize_and_tag(text, entity=None):

    tokens = re.findall(r"[\w']+|[.,!?;()\-]", text)
    tags = ["O"] * len(tokens)
    
    if entity:
        entity_tokens = re.findall(r"[\w']+|[.,!?;()\-]", entity)
        e_len = len(entity_tokens)
        
        for i in range(len(tokens) - e_len + 1):
            if tokens[i:i+e_len] == entity_tokens:
                tags[i] = "B-MOUNTAIN"
                for j in range(1, e_len):
                    tags[i+j] = "I-MOUNTAIN"
                    
    return {"tokens": tokens, "ner_tags": tags}


dataset = []


for template in positive_templates:
    for mountain in mountains:
        sentence = template.format(mountain=mountain)
        dataset.append(tokenize_and_tag(sentence, entity=mountain))


for neg in negative_sentences:
    dataset.append(tokenize_and_tag(neg, entity=None))


random.seed(42)
random.shuffle(dataset)


output_path = "data/train_ner_dataset.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=4)

print(json.dumps(dataset[0], indent=2))

{
  "tokens": [
    "The",
    "view",
    "from",
    "the",
    "top",
    "of",
    "Mount",
    "Rainier",
    "was",
    "absolutely",
    "breathtaking",
    "."
  ],
  "ner_tags": [
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "B-MOUNTAIN",
    "I-MOUNTAIN",
    "O",
    "O",
    "O",
    "O"
  ]
}
